# 💿 Day 02a: Disk Scheduling — FCFS, SSTF, SCAN, C-SCAN, LOOK

---

## 🎯 What You'll Master Today
- FCFS, SSTF, SCAN (Elevator), C-SCAN, LOOK, C-LOOK
- Total head movement calculation
- Visual comparison of all algorithms

---

## 🛗 The Elevator Analogy

Disk head movement = elevator in a building:
- **FCFS** = visit floors in request order (zigzag chaos)
- **SSTF** = always go to nearest requested floor (greedy)
- **SCAN** = go up to top, then down to bottom (elevator)
- **C-SCAN** = go up to top, jump to bottom, go up again (circular)
- **LOOK** = like SCAN but don't go to endpoints, reverse at last request

```
╔══════════════════════════════════════════════════════════╗
║              DISK SCHEDULING OVERVIEW                    ║
╠══════════════════════════════════════════════════════════╣
║  FCFS  — simple, high seek time                         ║
║  SSTF  — greedy, good avg but starvation possible       ║
║  SCAN  — elevator, fair, no starvation                  ║
║  C-SCAN — uniform wait time                              ║
║  LOOK  — SCAN but reverses at last request              ║
╚══════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# All Disk Scheduling Algorithms
# ============================================

def fcfs(requests: list[int], head: int) -> tuple[int, list[int]]:
    """First Come First Served — serve in arrival order."""
    order = [head] + requests
    movement = sum(abs(order[i] - order[i-1]) for i in range(1, len(order)))
    return movement, requests


def sstf(requests: list[int], head: int) -> tuple[int, list[int]]:
    """Shortest Seek Time First — greedy nearest."""
    remaining = requests[:]
    order = []
    current = head
    movement = 0
    
    while remaining:
        nearest = min(remaining, key=lambda r: abs(r - current))
        movement += abs(nearest - current)
        current = nearest
        order.append(nearest)
        remaining.remove(nearest)
    
    return movement, order


def scan(requests: list[int], head: int, disk_size: int, direction: str = "right") -> tuple[int, list[int]]:
    """SCAN (Elevator) — go to end, reverse, go to other end."""
    left = sorted([r for r in requests if r < head])
    right = sorted([r for r in requests if r >= head])
    
    if direction == "right":
        order = right + [disk_size - 1] + left[::-1]
    else:
        order = left[::-1] + [0] + right
    
    movement = 0
    current = head
    for pos in order:
        movement += abs(pos - current)
        current = pos
    
    return movement, [r for r in order if r in requests]


def look(requests: list[int], head: int, direction: str = "right") -> tuple[int, list[int]]:
    """LOOK — like SCAN but reverse at last request, not disk edge."""
    left = sorted([r for r in requests if r < head])
    right = sorted([r for r in requests if r >= head])
    
    if direction == "right":
        order = right + left[::-1]
    else:
        order = left[::-1] + right
    
    movement = 0
    current = head
    for pos in order:
        movement += abs(pos - current)
        current = pos
    
    return movement, order


def c_scan(requests: list[int], head: int, disk_size: int) -> tuple[int, list[int]]:
    """C-SCAN — go right to end, jump to 0, go right again."""
    left = sorted([r for r in requests if r < head])
    right = sorted([r for r in requests if r >= head])
    
    order = right + [disk_size - 1, 0] + left
    
    movement = 0
    current = head
    for pos in order:
        movement += abs(pos - current)
        current = pos
    
    return movement, [r for r in order if r in requests]


# ============================================
# Compare All Algorithms
# ============================================

requests = [98, 183, 37, 122, 14, 124, 65, 67]
head = 53
disk_size = 200

print(f"  Requests: {requests}")
print(f"  Initial Head: {head}")
print(f"  Disk Size: {disk_size}")
print(f"  {'═' * 55}")

algorithms = [
    ("FCFS",   lambda: fcfs(requests, head)),
    ("SSTF",   lambda: sstf(requests, head)),
    ("SCAN",   lambda: scan(requests, head, disk_size)),
    ("LOOK",   lambda: look(requests, head)),
    ("C-SCAN", lambda: c_scan(requests, head, disk_size)),
]

results = []
for name, func in algorithms:
    movement, order = func()
    results.append((name, movement, order))
    print(f"\n  {name}:")
    print(f"    Order: {head} → {' → '.join(map(str, order))}")
    print(f"    Total Head Movement: {movement} cylinders")

# Summary bar chart
print(f"\n  {'═' * 55}")
print("  COMPARISON SUMMARY")
print(f"  {'═' * 55}")
max_mov = max(r[1] for r in results)
for name, movement, _ in sorted(results, key=lambda x: x[1]):
    bar_len = int(movement / max_mov * 30)
    bar = '█' * bar_len + '░' * (30 - bar_len)
    print(f"    {name:<7} {bar} {movement}")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | SSTF vs SCAN? | SSTF: better avg seek, starvation risk. SCAN: fair, no starvation |
| 2 | Why C-SCAN over SCAN? | Uniform wait time — SCAN favors middle cylinders |
| 3 | LOOK vs SCAN? | LOOK reverses at last request (not disk edge). Less unnecessary travel |
| 4 | Which is used in practice? | Linux uses variants of C-LOOK (CFQ, deadline, BFQ schedulers) |
| 5 | Does SSD need disk scheduling? | No seek time in SSD. Simple FCFS/NOOP is fine for flash storage |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • FCFS: fair but worst performance                      │
## │  • SSTF: best avg but starvation at edges                │
## │  • SCAN/LOOK: elevator approach, balanced & fair         │
## │  • C-SCAN: uniform wait, used in real systems            │
## │  • SSDs don't need disk scheduling (no physical head)    │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **I/O Systems** — polling, interrupts, DMA & RAID